# 🎙️ Clone lessons from Google Drive — Colab (one click)

Everything lives in your Drive folder **`My Drive/FFIQ_voice/`**:

| Folder | What goes there |
|---|---|
| `voice_reference_A.wav` | the approved course voice (already there) |
| `input/` | lesson `.json` files waiting to be cloned (Claude puts them there) |
| `output/<lesson>/` | the cloned clips, saved one by one, plus a `_DONE.json` marker |
| `input/done/` | lesson files that finished |

**How to run:** `Runtime → Change runtime type → T4 GPU`, then **`Runtime → Run all`**.
Allow Google Drive access when asked. That's it.

If Colab disconnects, just **Run all** again: finished clips are skipped, so it
continues where it stopped. Your PC picks the results up automatically.

In [ ]:
# CELL 1 — Setup: voice-engine code + install. ~5-10 min.
!nvidia-smi -L || echo 'NO GPU: Runtime -> Change runtime type -> GPU, then re-run.'
import os
os.chdir('/content')   # re-running the cell must not nest the repo folder
USER, REPO, BRANCH = 'PrashantRMarathe', 'voice-replacer', 'add-course-video'
if os.path.isdir(REPO):
    !cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/{USER}/{REPO}.git
%cd {REPO}
# ONE pip call so the transformers<5 pin also holds for F5 (a separate f5-tts
# install upgraded transformers to v5 -> F5 failed -> silent XTTS fallback).
# f5-tts is LOCKED to 1.1.22: the version that made the approved voice.
!pip install -q coqui-tts f5-tts==1.1.22 'transformers<5' ffmpeg-python soundfile num2words
import importlib, importlib.metadata as md
try:
    importlib.import_module('f5_tts.api')
    F5_VERSION = md.version('f5-tts')
    print('F5-TTS version:', F5_VERSION, '(locked: 1.1.22)')
except Exception as e:
    for pkg in ('f5-tts', 'transformers', 'huggingface_hub', 'torch'):
        try: print(f'  {pkg}:', md.version(pkg))
        except Exception: print(f'  {pkg}: not installed')
    raise RuntimeError(f'STOP: F5-TTS did not import ({e}). Screenshot this for Claude.')
print('\n✅ Setup done.')

In [ ]:
# CELL 2 — Connect Google Drive (click Allow when asked)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, hashlib
BASE = '/content/drive/MyDrive/FFIQ_voice'
VOICE = f'{BASE}/voice_reference_A.wav'
for p in (BASE, f'{BASE}/input', f'{BASE}/output', VOICE):
    if not os.path.exists(p):
        raise RuntimeError(f'STOP: missing {p} in your Drive. Check the FFIQ_voice folder.')
os.makedirs(f'{BASE}/input/done', exist_ok=True)
VOICE_MD5 = hashlib.md5(open(VOICE, 'rb').read()).hexdigest()
JOBS = sorted(glob.glob(f'{BASE}/input/*.json'))
print('Voice clip:', os.path.basename(VOICE), VOICE_MD5[:12])
print('Lessons waiting:', [os.path.basename(j)[:-5] for j in JOBS] or 'none')
print('\n✅ Drive connected.')

In [ ]:
# CELL 3 — Clone every lesson in input/ (resumes after a disconnect)
import os, json, time, shutil, datetime, hashlib
os.environ['COQUI_TOS_AGREED'] = '1'
import config, tts
config.TTS_BACKEND = 'f5'
config.LANGUAGE = 'en'

if not JOBS:
    print('Nothing to do: put lesson .json files in FFIQ_voice/input/.')
else:
    backend = tts.get_backend()   # load the voice model ONCE for all lessons
    if backend.name != 'f5':
        raise RuntimeError('STOP: not using F5 (got ' + backend.name + '). Re-run Cell 1.')
    print('Voice engine:', backend.name, F5_VERSION)
    os.makedirs('/content/tmp_wav', exist_ok=True)
    t0 = time.time()
    for job in JOBS:
        name = os.path.basename(job)[:-5]
        out_dir = f'{BASE}/output/{name}'
        job_md5 = hashlib.md5(open(job, 'rb').read()).hexdigest()
        if os.path.exists(f'{out_dir}/_DONE.json'):
            prev = json.load(open(f'{out_dir}/_DONE.json'))
            # Same script already finished (Colab stopped before moving the file):
            # just file it under done/, don't clone again.
            if prev.get('script_md5') == job_md5:
                shutil.move(job, f'{BASE}/input/done/{name}.json')
                print(f'✅ {name} was already finished -> moved to input/done/'); continue
            # A NEW version of a finished lesson -> archive the earlier run.
            stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
            os.makedirs(f'{BASE}/output/_archive', exist_ok=True)
            shutil.move(out_dir, f'{BASE}/output/_archive/{name}_{stamp}')
        os.makedirs(out_dir, exist_ok=True)
        scripts = json.load(open(job, encoding='utf-8'))
        print(f'\n=== {name}: {len(scripts)} clips ===')
        for i, (key, text) in enumerate(scripts.items(), 1):
            out = f'{out_dir}/{key}.wav'
            if os.path.exists(out) and os.path.getsize(out) > 1000:
                print(f'  [{i}/{len(scripts)}] already done {key}'); continue
            tmp = f'/content/tmp_wav/{key}.wav'     # write locally, then copy:
            backend.synth_to_file(text=text, speaker_wav=VOICE,   # Drive never sees
                                  language=config.LANGUAGE, out_path=tmp)  # half a file
            shutil.copy(tmp, out + '.part'); os.replace(out + '.part', out)
            print(f'  [{i}/{len(scripts)}] {key}  ({time.time()-t0:.0f}s)')
        json.dump({'lesson': name, 'clips': len(scripts), 'keys': list(scripts),
                   'engine': backend.name, 'f5_version': F5_VERSION,
                   'voice': os.path.basename(VOICE), 'voice_md5': VOICE_MD5,
                   'script_md5': job_md5,
                   'finished': datetime.datetime.now().isoformat(timespec='seconds')},
                  open(f'{out_dir}/_DONE.json', 'w'), indent=2)
        shutil.move(job, f'{BASE}/input/done/{name}.json')
        print(f'✅ {name} finished -> output/{name}/')
    print('\n✅ All lessons cloned.')

In [ ]:
# CELL 4 — Make sure everything is saved to Drive
from google.colab import drive
drive.flush_and_unmount()
print('✅ Saved to Drive. Your PC will pick the results up automatically.')